# 01. Tìm Hiểu Dữ Liệu và Tiền Xử Lý Dữ Liệu
**Học phần:** Khai thác dữ liệu - Nhóm 12  
**Đề tài 16:** Phân nhóm quốc gia theo các yếu tố tạo nên mức độ hạnh phúc (World Happiness Report)  
---
### Đối chiếu với Mục 1.2 - Pipeline bắt buộc của đồ án:
- [x] **Bước 1:** Xác định bài toán và câu hỏi khai thác dữ liệu.
- [x] **Bước 2:** Tìm hiểu dữ liệu (nguồn gốc, ý nghĩa biến, đơn vị đo, ghép 5 năm).
- [x] **Bước 3:** Đánh giá chất lượng dữ liệu (giá trị thiếu, trùng lặp, tính nhất quán schema).
- [x] **Bước 4 (Phần 1):** Tiền xử lý dữ liệu (chuẩn hóa tên cột, xử lý missing, hợp nhất dữ liệu).

## [Bước 1/11] Xác định bài toán và câu hỏi khai thác dữ liệu
1. **Bối cảnh & Mục tiêu:**
   - Đề tài tập trung vào bài toán **Gom cụm (Clustering / Unsupervised Learning)** để phân nhóm các quốc gia trên thế giới.
   - Mục tiêu: Phát hiện các hình thái phân hóa chất lượng sống dựa trên 6 yếu tố cơ sở (GDP, Hỗ trợ xã hội, Sức khỏe/Tuổi thọ, Quyền tự do, Sự hào phóng, Nhận thức tham nhũng).
2. **Câu hỏi khai thác dữ liệu cụ thể:**
   - Có thể phân chia các quốc gia trên thế giới thành những nhóm (cụm) đặc trưng nào?
   - Các nhóm quốc gia khác biệt rõ rệt nhất ở những yếu tố nào (ví dụ: kinh tế vs thể chế xã hội)?
   - Nhóm quốc gia nào ở tình trạng báo động và cần sự hỗ trợ/viện trợ quốc tế?
3. **Quy tắc vàng (Bắt buộc theo hướng dẫn đề tài 16):**
   - Tuyệt đối **KHÔNG** đưa thuộc tính `happiness_score` hoặc `happiness_rank` vào làm biến đầu vào để gom cụm. Chúng chỉ được dùng để hậu kiểm ở Notebook 04.

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np

project_root = Path('..').resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.data.data_loader import load_raw_data, load_year_data, standardize_columns
print('Project root:', project_root)

## [Bước 2/11] Tìm hiểu dữ liệu (Data Understanding)
- **Nguồn dữ liệu:** Kaggle - World Happiness Report (2015 - 2019).
- **Nội dung:** Khảo sát hàng năm về chất lượng cuộc sống dựa trên thang đo Gallup World Poll (Cantril Ladder từ 0 đến 10).
- **Nhiệm vụ:** Tải và kiểm tra cấu trúc 5 tệp dữ liệu tương ứng các năm từ 2015 đến 2019.

In [ ]:
raw_data = load_raw_data(raw_dir=project_root / 'data' / 'raw')
for year, df in raw_data.items():
    print(f'Năm {year}: {df.shape[0]} dòng, {df.shape[1]} cột')
    print(f'   Các cột ban đầu: {list(df.columns[:5])} ...')

In [ ]:
# Xem cấu trúc chi tiết và kiểu dữ liệu của năm 2019
raw_data[2019].info()
raw_data[2019].head()

## [Bước 3/11] Đánh giá chất lượng dữ liệu (Data Quality Assessment)
Kiểm tra các khía cạnh:
1. Sự khác biệt về tên cột và định dạng dữ liệu qua 5 năm.
2. Kiểm tra giá trị thiếu (Missing values / NaN).
3. Kiểm tra dữ liệu trùng lặp (Duplicate rows / Country names).

In [ ]:
# 1. Đánh giá tính nhất quán của tên cột
for year, df in raw_data.items():
    print(f'\n--- Danh sách cột năm {year} sau chuẩn hóa ---')
    print(list(df.columns))

In [ ]:
# 2. Kiểm tra giá trị thiếu (Missing values) trong từng năm
for year, df in raw_data.items():
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    print(f'Năm {year}: {len(missing)} cột có giá trị thiếu')
    if len(missing) > 0:
        print(missing)

In [ ]:
# 3. Kiểm tra bản ghi trùng lặp (Duplicates)
for year, df in raw_data.items():
    dup_count = df.duplicated(subset=['country']).sum()
    print(f'Năm {year}: {dup_count} quốc gia bị trùng lặp')

## [Bước 4/11 - Phần 1] Tiền xử lý dữ liệu (Data Preprocessing)
- Chuẩn hóa định dạng snake_case cho toàn bộ thuộc tính.
- Hợp nhất dữ liệu các năm thành bảng trung gian thống nhất.
- Lưu trữ tệp sạch vào `data/interim/happiness_merged.csv`.

In [ ]:
merged_df = pd.concat(raw_data.values(), ignore_index=True)
interim_path = project_root / 'data' / 'interim' / 'happiness_merged.csv'
merged_df.to_csv(interim_path, index=False)
print(f'Đã lưu bảng dữ liệu trung gian: {merged_df.shape} vào {interim_path}')